# geo0264 – Pseudoranges: compute your own GNSS position

**Unit:** geo0260 GNSS, RTK and SAPOS · **Slides:** `geo0260_GNSS` (sections *GNSS*, *DGNSS / RTK*, *SAPOS*) · **Data:** `data/geo0264/` or your group's field data · **Optional:** for everyone who wants to look inside the receiver

A GNSS receiver does not measure positions. It measures **pseudoranges**: travel times of satellite signals, multiplied by the speed of light and spoiled by its own clock error. In this notebook you turn pseudoranges into a position yourself and check it against a position you know.

The notebook works with three data sources (§0): **synthetic** pseudoranges on real GPS orbits, where you switch every error on and off; **SAPOS** reference stations, measured on 21 October 2026, known to the millimetre; and **your own** raw data from the field session, compared with your RTK fix. The code of §2–§5 is the same for all three.

You will learn to
1. set up and solve the pseudorange equations with **four unknowns** (X, Y, Z, receiver clock),
2. judge a solution against a known point (east, north, up errors),
3. reduce the errors step by step: elevation mask, troposphere, **ionosphere-free combination**,
4. explain the remaining offset of about 1 m (ETRS89 vs WGS 84) and how a **reference station** (DGNSS) removes it.

Levels: §1–§3 core (about 2 h), §4–§5 stretch.

## 0. Setup: choose your data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer
import geo0264_gnss_prep as prep     # helper module in this folder: data sources and orbits

C, F1, F2 = prep.C, prep.F1, prep.F2   # speed of light [m/s], GPS L1 and L2 frequencies [Hz]
DATA = Path("data") / "geo0264"

SOURCE = "sapos"          # "synthetic" | "sapos" | "own"
OWN_FILE = Path.home() / "shared" / "data" / "opengeo" / "geo0264" / "group01" / "geo0264_group01.csv"
POINT = None              # own data: name of one surveyed point (None = the first one)

**What the sources contain**

| `SOURCE` | rover (`ROVER`) | reference station (`BASE`, §5) | truth |
|---|---|---|---|
| `"synthetic"` | invented receiver at the Green Fab Lab, real GPS orbits of 21 Oct 2026, errors from `ERRORS` | invented second receiver at Wesel, sharing the satellite and atmosphere errors | exact |
| `"sapos"` | SAPOS station 2579 Viersen | SAPOS station 2578 Wesel (48 km) | station coordinates (ETRS89) |
| `"own"` | one of your surveyed points (`POINT`) | SAPOS station 2578 Wesel, same day | your RTK fix + pole height (ETRS89) |

For `"synthetic"` you choose the errors (in metres at the zenith, 0 switches a term off):

In [ ]:
ERRORS = dict(prep.ERRORS)            # defaults, see the table printed below
# ERRORS["iono_zenith_m"] = 0         # e.g. switch the ionosphere off
pd.Series(prep.ERRORS, name="default")

In [ ]:
sapos_tab, sapos_ref = prep.load(DATA / "geo0264_sapos_2026-10-21.csv")

if SOURCE == "sapos":
    tab, ref, ROVER, BASE = sapos_tab, sapos_ref, "2579", "2578"
elif SOURCE == "synthetic":
    orbits = sapos_tab[sapos_tab["site"] == "2578"]                     # real orbits, pseudoranges replaced
    tab, ref = prep.synthetic(orbits, lon=6.548834, lat=51.498069, h=72.0, errors=ERRORS,
                              base=(6.6081, 51.6648, 140.0), seed=1)   # base near Wesel
    ROVER, BASE = "rover", "base"
elif SOURCE == "own":
    tab, ref = prep.load(OWN_FILE)
    points = [s for s in ref["site"] if not s.isdigit()]
    ROVER = POINT or points[0]
    BASE = next((s for s in ref["site"] if s.isdigit()), None)          # SAPOS station, if included
else:
    raise ValueError(SOURCE)

tab["time"] = pd.to_datetime(tab["time"])
REF = ref.set_index("site")
obs = tab[tab["site"] == ROVER]
X_ref = REF.loc[ROVER, ["X_m", "Y_m", "Z_m"]].to_numpy(float)
print(f"source {SOURCE}: rover {ROVER}, base {BASE}, {obs['time'].nunique()} epochs, "
      f"{obs['time'].min()} – {obs['time'].max()}")
ref

## 1. Warm-up: the solver

### 1.1 The equations
For satellite *i* at position $(X_i, Y_i, Z_i)$ the receiver measures the pseudorange

$$\rho_i = \sqrt{(X_i - X)^2 + (Y_i - Y)^2 + (Z_i - Z)^2} + c\,\delta t_r$$

The unknowns are the receiver position $(X, Y, Z)$ and the receiver clock error $\delta t_r$; we solve for $b = c\,\delta t_r$ in metres. Four unknowns need at least four satellites. The equation is not linear, so we start with a guess (the Earth's centre), linearise and improve the guess until it no longer changes (Gauss–Newton, as on the slide *Try it*).

`make_test_case` takes the real satellites of one epoch and invents the pseudoranges for a receiver at the Green Fab Lab, so the true position is known exactly.

In [ ]:
TEST_ORBITS = sapos_tab[sapos_tab["site"] == "2578"]


def make_test_case(n_sat=None, clock_us=100.0, noise_m=2.0, min_elev=10.0, seed=1):
    # One epoch of invented pseudoranges on real orbits: (sat, rho, X_true).
    # n_sat: use only n randomly chosen satellites; min_elev: only satellites higher than this.
    rng = np.random.default_rng(seed)
    t = rng.choice(TEST_ORBITS["time"].unique())
    errs = {k: 0.0 for k in prep.ERRORS} | {"receiver_clock_us": clock_us, "noise_m": noise_m}
    syn, r = prep.synthetic(TEST_ORBITS[TEST_ORBITS["time"] == t], errors=errs, seed=seed, mask=min_elev)
    if n_sat is not None:
        syn = syn.sample(min(n_sat, len(syn)), random_state=seed)
    sat = syn[["x_m", "y_m", "z_m"]].to_numpy()
    rho = (syn["C1_m"] + syn["sat_clock_m"] - syn["tgd_m"]).to_numpy()   # satellite clock corrected
    return sat, rho, r.loc[0, ["X_m", "Y_m", "Z_m"]].to_numpy(float)


sat, rho, X_true = make_test_case()
pd.DataFrame(np.c_[sat, rho] / 1000, columns=["X_km", "Y_km", "Z_km", "rho_km"]).round(1)

### 1.2 The maths: linearise, solve, repeat
What your solver will do, step by step. Write $\mathbf{p} = (X, Y, Z, b)$ for the unknowns, $\mathbf{s}_i$ for the position of satellite *i* and $r_i(\mathbf{x}) = \lVert \mathbf{s}_i - \mathbf{x} \rVert$ for the geometric range, so that $\rho_i = r_i(\mathbf{x}) + b + \varepsilon_i$ ($\varepsilon_i$: noise).

**Linearise.** Around a guess $\mathbf{p}_0 = (\mathbf{x}_0, b_0)$, a first-order Taylor expansion gives

$$\underbrace{\rho_i - \big(r_i(\mathbf{x}_0) + b_0\big)}_{\ell_i \;=\; \text{observed} - \text{computed}} \;\approx\; \frac{\partial r_i}{\partial \mathbf{x}}\bigg|_{\mathbf{x}_0} \Delta\mathbf{x} + \Delta b$$

The right-hand side is linear in the corrections $\Delta\mathbf{p} = (\Delta\mathbf{x}, \Delta b)$.

**The partial derivatives.** Differentiating the square root,

$$\frac{\partial r_i}{\partial X} = \frac{X - X_i}{r_i}, \quad \text{likewise for } Y, Z: \qquad \frac{\partial r_i}{\partial \mathbf{x}} = \frac{(\mathbf{x} - \mathbf{s}_i)^T}{r_i} = \mathbf{e}_i^T$$

$\mathbf{e}_i$ is the **unit vector from satellite *i* to the receiver**. Moving the receiver along the line of sight changes the range one to one; moving it sideways hardly changes it at all. The derivative with respect to $b$ is 1. Stacking all satellites gives the $n \times 4$ design matrix

$$A = \begin{bmatrix} \mathbf{e}_1^T & 1 \\ \vdots & \vdots \\ \mathbf{e}_n^T & 1 \end{bmatrix}, \qquad A\,\Delta\mathbf{p} \approx \boldsymbol{\ell}$$

**Least squares.** With more than four satellites and noisy ranges, $A\,\Delta\mathbf{p} = \boldsymbol{\ell}$ has no exact solution. Least squares picks the $\Delta\mathbf{p}$ that minimises $\lVert \boldsymbol{\ell} - A\,\Delta\mathbf{p} \rVert^2$; setting the gradient to zero gives the normal equations

$$A^T A\,\Delta\mathbf{p} = A^T \boldsymbol{\ell} \quad\Rightarrow\quad \Delta\mathbf{p} = (A^T A)^{-1} A^T \boldsymbol{\ell}$$

`np.linalg.lstsq` solves the same problem in a numerically more stable way, without inverting $A^T A$.

**Repeat.** $\mathbf{p}_{k+1} = \mathbf{p}_k + \Delta\mathbf{p}_k$; recompute $r_i$, $A$ and $\boldsymbol{\ell}$ at the new guess and stop when $\lVert \Delta\mathbf{p} \rVert$ is below a millimetre. This is the **Gauss–Newton** method: it uses only first derivatives ($A^T A$ stands in for the full second-derivative matrix). Starting at the Earth's centre works because the satellites are about 26 000 km away: a guess 6 400 km off hardly changes the directions $\mathbf{e}_i$, so a few iterations are enough.

**Why the clock does no harm.** $b$ enters every equation with the coefficient 1 (the column of ones). An offset common to all pseudoranges goes completely into $b$ and leaves the position untouched; you test this in §1.5.

**Why at least four satellites.** Four unknowns need $\operatorname{rank}(A) = 4$: at least four satellites in a non-degenerate geometry. *How good* the geometry is, PDOP measures (§1.4).

### 1.3 Write the solver
**Exercise:** write `solve_position(sat, rho)`. It returns the array `[X, Y, Z, b]` (metres, $b = c\,\delta t_r$) and the design matrix `A` of the last iteration (we need it for the DOP).

Hints (the maths is in §1.2): in every iteration compute the ranges `r` from the current guess to all satellites; the rows of `A` are the unit vectors from the satellites to the receiver, plus a column of ones for the clock; solve `A @ dx = rho - (r + b)` with `np.linalg.lstsq(A, ..., rcond=None)` and stop when `dx` is smaller than a millimetre.

In [ ]:
def solve_position(sat, rho, max_iter=10):
    # Least-squares position from pseudoranges: returns x = [X, Y, Z, b] and the design matrix A.
    # YOUR CODE HERE


In [ ]:
x, A = solve_position(sat, rho)
print("position error [m]:", round(float(np.linalg.norm(x[:3] - X_true)), 2))
print("receiver clock [µs]:", round(x[3] / C * 1e6, 3))

### 1.4 Geometry: PDOP
If every pseudorange has the same, independent noise $\sigma$, the covariance of the solution is $\Sigma = \sigma^2 (A^T A)^{-1} = \sigma^2 Q$. The position dilution of precision follows from the design matrix: $Q = (A^T A)^{-1}$, $\mathrm{PDOP} = \sqrt{Q_{XX} + Q_{YY} + Q_{ZZ}}$. Position error ≈ PDOP × range error.

In [ ]:
def pdop(A):
    Q = np.linalg.inv(A.T @ A)
    return float(np.sqrt(np.trace(Q[:3, :3])))


pdop(A)

### 1.5 Play
**Predict first, then run.** Change one parameter at a time; every case runs 50 times with different epochs (satellite geometries):
* `clock_us=1000` (the receiver clock is 1 ms off, i.e. 300 km of range): does the error change?
* `noise_m=0` and `noise_m=10`;
* `n_sat=4`, all satellites;
* `min_elev=35` (only satellites high in the sky).

In [ ]:
rows = []
for kw in [{}, {"clock_us": 1000}, {"noise_m": 0}, {"noise_m": 10}, {"n_sat": 4}, {"min_elev": 35}]:
    errs, dops = [], []
    for seed in range(50):
        s_, r_, X_ = make_test_case(seed=seed, **kw)
        if len(r_) < 4:
            continue
        x_, A_ = solve_position(s_, r_)
        errs.append(np.linalg.norm(x_[:3] - X_)), dops.append(pdop(A_))
    rows.append({"change": str(kw) if kw else "default", "median error [m]": np.median(errs),
                 "median PDOP": np.median(dops), "runs": len(errs)})
pd.DataFrame(rows).round(2)

**Questions:** Why does the clock error not change the result at all? Why do 4 satellites, or only high satellites, give much larger errors, although they are enough to solve the equations?

✍️ *Your answer:*

## 2. Your data source

### 2.1 The table
All three sources give the same table, one row per satellite and epoch:

| column | meaning |
|---|---|
| `site` | station or point name |
| `time` | epoch, GPS time |
| `prn` | satellite (G01 … G32) |
| `x_m`, `y_m`, `z_m` | satellite position at transmission, Earth-fixed (ECEF, WGS 84), from the broadcast ephemeris |
| `sat_clock_m` | satellite clock error × c [m], **add** it to the pseudorange |
| `tgd_m` | group delay × c [m], **subtract** it for single-frequency L1 |
| `C1_m` | pseudorange on L1 (C/A code) [m] |
| `C2_m`, `c2_code` | pseudorange on L2 [m] and its signal (`C2W` SAPOS, `C2L` Reach RS2+) |

The reference table `ref` holds the true antenna positions (geocentric X, Y, Z). SAPOS data: GPPS archive, Geobasis NRW, Datenlizenz Deutschland – Zero – Version 2.0.

**Look first:** how many satellites per epoch? How large are the pseudoranges (km)? How large are the satellite clock corrections?

In [ ]:
print(obs.groupby("time")["prn"].count().describe().round(1))
obs.assign(C1_km=obs["C1_m"] / 1000)[["C1_km", "sat_clock_m", "tgd_m"]].describe().round(1)

### 2.2 Errors in east, north, up
A difference of 3 m in X tells us little. `enu` turns a geocentric difference into local east, north and up components at the reference point.

In [ ]:
TO_LLH = Transformer.from_crs(4936, 4937, always_xy=True)


def enu(dxyz, X_ref):
    # Geocentric difference vector(s) -> east, north, up [m] at X_ref.
    lon, lat, _ = TO_LLH.transform(*X_ref)
    la, lo = np.radians(lat), np.radians(lon)
    R = np.array([[-np.sin(lo), np.cos(lo), 0],
                  [-np.sin(la) * np.cos(lo), -np.sin(la) * np.sin(lo), np.cos(la)],
                  [np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)]])
    return np.asarray(dxyz) @ R.T

### 2.3 One epoch
**Exercise:** take the first epoch, correct the L1 pseudoranges for the satellite clock and the group delay (`C1_m + sat_clock_m - tgd_m`), solve with your function and print the error in east, north, up, the receiver clock in µs and the PDOP.

In [ ]:
# YOUR CODE HERE


## 3. All epochs, and where the metres come from

### 3.1 All epochs
`solve_all` runs your solver for every epoch. The argument `correct(g)` is a function that returns, for the rows `g` of one epoch, the corrected pseudoranges and a mask of the satellites to use.

In [ ]:
def solve_all(obs, X_ref, correct):
    rows = []
    for t, g in obs.groupby("time"):
        rho, use = correct(g)
        sat = g[["x_m", "y_m", "z_m"]].to_numpy()
        if use.sum() < 4:
            continue
        x, A = solve_position(sat[use], rho[use])
        e, n, u = enu(x[:3] - X_ref, X_ref)
        rows.append({"time": t, "E": e, "N": n, "U": u, "clock_us": x[3] / C * 1e6, "PDOP": pdop(A),
                     "n_sat": int(use.sum())})
    return pd.DataFrame(rows).set_index("time")


def l1_raw(g):
    rho = (g["C1_m"] + g["sat_clock_m"] - g["tgd_m"]).to_numpy()
    return rho, np.ones(len(g), bool)


def summary(res, label):
    s = res[["E", "N", "U"]].agg(["mean", "std"]).round(2)
    s.index = [f"{label} {i}" for i in s.index]
    return s


def plot_enu(res, title):
    if res.index.nunique() > 50 and (res.index.max() - res.index.min()) > pd.Timedelta("2h"):
        ax = res[["E", "N", "U"]].plot(figsize=(10, 3.5), title=title, lw=1)     # long series: time plot
        ax.set_ylabel("error [m]")
    else:                                                                       # short: scatter around the truth
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].scatter(res["E"], res["N"], s=8), ax[0].set_aspect("equal"), ax[0].set_title(title)
        ax[0].set_xlabel("east error [m]"), ax[0].set_ylabel("north error [m]")
        ax[1].plot(res["U"].to_numpy(), lw=1), ax[1].set_ylabel("up error [m]"), ax[1].set_xlabel("epoch")
        for a in ax:
            a.axhline(0, color="k", lw=0.5)
    return ax


raw = solve_all(obs, X_ref, l1_raw)
plot_enu(raw, f"{ROVER}, L1, no corrections")
summary(raw, "L1 raw")

**Look:** which component is worst? Is the error random or is there a systematic offset?

### 3.2 Elevation mask and troposphere
Signals from satellites low above the horizon travel a long way through the atmosphere and are often reflected (multipath). The troposphere delays every signal by about 2.3 m at the zenith, roughly $2.3\,\mathrm{m} / \sin(\text{elevation})$ for lower satellites.

**Exercise:** write `elevation(sat, X_ref)` (elevation angles in degrees of all satellites seen from `X_ref`), then a correction function `l1_mask_tropo(g)` that subtracts the troposphere and uses only satellites above 10°. Compare with the raw solution.

Hint: elevation = angle between the line of sight `sat - X_ref` and the local *up* direction; for a sphere, *up* is simply `X_ref / |X_ref|` (good enough here).

In [ ]:
def elevation(sat, X_ref):
    # YOUR CODE HERE


def l1_mask_tropo(g, X_ref=X_ref, mask=10):
    # YOUR CODE HERE


masked = solve_all(obs, X_ref, l1_mask_tropo)
plot_enu(masked, f"{ROVER}, L1, mask 10°, troposphere")
pd.concat([summary(raw, "L1 raw"), summary(masked, "L1 mask+tropo")])

**Questions:** Why did the height improve most? Which error source from the slides has not been corrected yet?

✍️ *Your answer:*

## 4. Stretch: the ionosphere-free combination

The ionosphere delays the code by an amount proportional to $1/f^2$. With two frequencies the delay can be eliminated:

$$\rho_{IF} = \frac{f_1^2\,\rho_1 - f_2^2\,\rho_2}{f_1^2 - f_2^2}$$

Use `C1_m` and `C2_m`, add `sat_clock_m` (no TGD: the broadcast clock refers to exactly this combination), subtract the troposphere and keep the 10° mask.

**Exercise:** write `iono_free(g)` and compare all three solutions. Which solution is more precise (std), which more accurate (mean)?

In [ ]:
def iono_free(g, X_ref=X_ref, mask=10):
    # YOUR CODE HERE


ifree = solve_all(obs, X_ref, iono_free)
pd.concat([summary(raw, "L1 raw"), summary(masked, "L1 mask+tropo"), summary(ifree, "iono-free")])

**A last metre (real data only).** With SAPOS or your own data, even the ionosphere-free solution is off by about 0.7 m east and 0.6 m north on average. Hint: the satellite orbits are given in WGS 84, the reference coordinates in ETRS89 (slide *What coordinates does SAPOS give you?*, geo0220). Compute the length and direction of the mean offset. Why is there no such offset with `SOURCE = "synthetic"`?

In [ ]:
# YOUR CODE HERE


✍️ *Your answer:*

## 5. Stretch: DGNSS with a reference station

The reference station `BASE` knows its true position, so for every satellite it can compute the error of its own pseudorange:

$$\Delta_i = |\mathbf{X}_i - \mathbf{X}_{\mathrm{base}}| - \rho_{i,\mathrm{base}}$$

Adding $\Delta_i$ to the rover's pseudorange of the same satellite at the same epoch removes the errors both share. Use corrected L1 pseudoranges (clock and TGD) and the 10° mask at the rover, but **no** troposphere model and **no** ionosphere-free combination: the correction should take care of them.

**Exercise:** compute the DGNSS solution for all common epochs and compare it with the rover's own L1 solution with mask and troposphere. (`merge` on `time` and `prn` gives you the satellites both see.)

In [ ]:
if BASE is None:
    raise SystemExit("no reference station in this data set")
base = tab[tab["site"] == BASE]
X_base = REF.loc[BASE, ["X_m", "Y_m", "Z_m"]].to_numpy(float)
print(f"baseline {BASE} – {ROVER}: {np.linalg.norm(X_ref - X_base) / 1000:.1f} km")

# YOUR CODE HERE
pd.concat([summary(masked, f"{ROVER} alone"), summary(dg, f"{ROVER} DGNSS")])

**Questions:** What happened to the 1 m offset of §4? In which reference frame is the DGNSS result? Why is the DGNSS solution still noisy at the decimetre level, and what does RTK do differently?

✍️ *Your answer:*

## 6. Wrap-up
1. Why is the receiver clock error harmless, although 1 ms means 300 km?
2. Rank the error sources you met by size, with the numbers from your own results.
3. Your phone shows ±4 m. Which of today's corrections does it probably use, which not?
4. Try another source (§0) and compare. With `"synthetic"`, switch single error terms off (`ERRORS`): which step of §3–§5 removes which error?
5. Optional: convert your mean ionosphere-free position to EPSG:25832 and DHHN2016 (`Transformer.from_crs(4936, "EPSG:25832+7837")`; mind the GCG2016 grid) and compare with the reference.

✍️ *Your answer:*